# Adapting a manifold with `AsVectorSpace`

`AsVectorSpace<Class>` is an opt-in C++ adapter for applications that deliberately need additive operations on a manifold-only type. This notebook explains why cumulative calibration splines need the adapter, the operations it defines, and the limits of the approximation.

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/AsVectorSpace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [ ]:
import gtsam
import numpy as np

## Why the adapter exists

A GTSAM manifold supplies `Local(origin, other)` and `Retract(origin, tangent)`, which are enough for nonlinear optimization. A Lie group additionally supplies a meaningful identity, composition, and inverse. Cumulative Lie-group splines use those group operations to form and accumulate relative control-point increments.

Camera calibration classes are manifolds but intentionally are not Lie groups: composing two intrinsic calibration matrices is not a physically meaningful operation. Nevertheless, an application may want a locally smooth zoom or calibration trajectory. `AsVectorSpace<Cal3_S2>` makes that application-level approximation explicit instead of changing the semantics of `Cal3_S2` itself.

## Operations chosen by the adapter

Let $e$ be the default-constructed wrapped value and define coordinates $\phi(x)=\operatorname{Local}(e,x)$. The adapter defines

$$x + y = \operatorname{Retract}(x,\phi(y)), \qquad x + v = \operatorname{Retract}(x,v),$$

with subtraction and negation obtained by negating the corresponding tangent coordinates. Its `vector()` method returns $\phi(x)$. These definitions satisfy GTSAM's additive vector-space interface for the supported calibration manifolds.

> **Modeling warning:** this is a chosen local affine embedding. It does not prove that the wrapped manifold has a natural or globally valid vector-space structure. Results can depend on the default value used as $e$, and large excursions can leave the region where the coordinates are a useful approximation.

In [ ]:
identity = gtsam.Cal3_S2()
calibration = gtsam.Cal3_S2(500.0, 510.0, 0.0, 320.0, 240.0)
coordinates = identity.localCoordinates(calibration)
reconstructed = identity.retract(coordinates)

print("Coordinates about the default calibration:", coordinates)
print("Local/Retraction round trip:", reconstructed.equals(calibration, 1e-9))

## C++ usage

The adapter is a header-only C++ template and is not currently exposed as a Python wrapper. It inherits the wrapped class, so existing calibration accessors remain available.

```cpp
using Calibration = AsVectorSpace<Cal3_S2>;
Calibration first(Cal3_S2(500.0, 510.0, 0.0, 320.0, 240.0));
Calibration second(Cal3_S2(520.0, 525.0, 0.0, 320.0, 240.0));
Calibration midpoint = first + 0.5 * (second - first).vector();
```

Use it as a cumulative-spline control-point type only when this affine calibration model is an intentional part of the application.

## Combining pose and calibration

`CartesianProduct<A, B>` is a descriptive alias for GTSAM's tested `ProductLieGroup<A, B>`. It can combine a physical pose group with an explicitly adapted calibration space:

```cpp
using CameraState = CartesianProduct<Pose3, AsVectorSpace<Cal3_S2>>;
CumulativeSplineTrajectory<CameraState> cameraTrajectory;
```

The pose component uses its natural Lie-group operations; the calibration component uses the chosen local affine embedding described above.

## When to use it

Use `AsVectorSpace` when the wrapped manifold has useful local coordinates, the trajectory remains near the chosen default value, and the affine interpretation is acceptable for the application. Prefer a domain-specific Lie group or a direct manifold interpolation model when composition has physical meaning or global behavior matters.

## Source

- [AsVectorSpace.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/AsVectorSpace.h)
- [CartesianProduct.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/CartesianProduct.h)
- [Cumulative spline notebook](../../basis/doc/CumulativeSplineTrajectory.ipynb)